# Firestore Collection Explorer

Sample records from the `extracted` and `analysis` collections to understand their structure, key presence, and value distributions.

In [1]:
import json
import os

import pandas as pd
from dotenv import load_dotenv
from google.cloud import firestore

load_dotenv()

if os.path.isfile("vk-linkedin-master-service-account.json"):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "vk-linkedin-master-service-account.json"

db = firestore.Client(project="vk-linkedin", database="linkedin")
print("Firestore connected")

Firestore connected


## 1. Sample `extracted` collection — first 10 records

In [2]:
SAMPLE_SIZE = 100

extracted_docs = [doc.to_dict() for doc in db.collection("extracted").limit(SAMPLE_SIZE).stream()]
print(f"Fetched {len(extracted_docs)} docs from 'extracted'")


Fetched 100 docs from 'extracted'


In [3]:
# Pretty-print the first record in full
print(json.dumps(extracted_docs[0], indent=2, default=str))

{
  "my_id": "2604",
  "connect": null,
  "memberDistance": {
    "personId": 3294,
    "memberDistance": "DISTANCE_3",
    "id": 3271
  },
  "isLastMessageIncoming": false,
  "messagesInfo": null,
  "invitedDate": null,
  "action_name": "",
  "campaign_id": 9,
  "profileUrl": "https://www.linkedin.com/in/-jananova-/",
  "lhId": 3294,
  "resultCreatedAt": null,
  "fullName": "Jana Nov\u00e1",
  "id": "-jananova-",
  "externalIds": [
    {
      "personId": 3294,
      "type": "member-id",
      "externalId": "483788075",
      "memberId": 483788075,
      "sentAtToPAS": "2024-03-21T10:21:10.617Z",
      "createdAt": "2021-01-27T01:35:38.026Z",
      "id": 10642
    },
    {
      "personId": 3294,
      "type": "public-id",
      "externalId": "-jananova-",
      "sentAtToPAS": "2024-03-21T10:21:10.617Z",
      "createdAt": "2021-01-27T01:35:38.026Z",
      "id": 10643
    },
    {
      "personId": 3294,
      "type": "li-hash-id",
      "externalId": "ACoAABzWBSsByZO8TXT3MpPPzsM-isZX

## 2. Key frequency in `extracted` — how often each top-level key appears (100 docs)

In [4]:
FREQ_SAMPLE = 100

freq_docs = [doc.to_dict() for doc in db.collection("extracted").limit(FREQ_SAMPLE).stream()]
total = len(freq_docs)
print(f"Fetched {total} docs for key-frequency analysis")

from collections import Counter

key_counts = Counter()
for doc in freq_docs:
    key_counts.update(doc.keys())

freq_df = pd.DataFrame(
    [(k, v, f"{100*v/total:.1f}%") for k, v in key_counts.most_common()],
    columns=["key", "count", "presence"],
)
freq_df

Fetched 100 docs for key-frequency analysis


,key,count,presence
0,my_id,100,100.0%
1,connect,100,100.0%
2,memberDistance,100,100.0%
3,isLastMessageIncoming,100,100.0%
4,messagesInfo,100,100.0%
5,invitedDate,100,100.0%
6,action_name,100,100.0%
7,campaign_id,100,100.0%
8,profileUrl,100,100.0%
9,lhId,100,100.0%


## 3. Nested key inspection — sub-keys inside complex fields

In [5]:
NESTED_KEYS = ["miniProfile", "currentPosition", "memberDistance", "externalIds", "positions", "email", "skills", "educations"]

sample = extracted_docs[0]  # use first record from earlier fetch

for key in NESTED_KEYS:
    val = sample.get(key)
    if val is None:
        print(f"\n[{key}] — NOT PRESENT in this record")
        continue
    snippet = json.dumps(val, indent=2, default=str)
    # Truncate long outputs for readability
    if len(snippet) > 800:
        snippet = snippet[:800] + "\n  ... (truncated)"
    print(f"\n[{key}]\n{snippet}")


[miniProfile]
{
  "lastName": "Nov\u00e1",
  "customLastNameOutdated": false,
  "originalSentAtToPAS": "2024-03-21T10:21:10.617Z",
  "createdAt": "2021-01-27T01:35:38.029Z",
  "avatar": "https://media-exp1.licdn.com/dms/image/C5603AQHhDykpHLxCRA/profile-displayphoto-shrink_800_800/0/1575895625362?e=1617235200&v=beta&t=nC0zZUdyVc1QRrEfQKl1-oWsdhj-1PUL3t2UNBkn-74",
  "customFirstName": null,
  "id": 3271,
  "customFirstNameOutdated": false,
  "personId": 3294,
  "originalLastName": "Nov\u00e1",
  "customLastName": null,
  "headline": "Market Access, Healthcare & Innovation",
  "originalFirstName": "Jana",
  "firstName": "Jana"
}

[currentPosition] — NOT PRESENT in this record

[memberDistance]
{
  "personId": 3294,
  "memberDistance": "DISTANCE_3",
  "id": 3271
}

[externalIds]
[
  {
    "personId": 3294,
    "type": "member-id",
    "externalId": "483788075",
    "memberId": 483788075,
    "sentAtToPAS": "2024-03-21T10:21:10.617Z",
    "createdAt": "2021-01-27T01:35:38.026Z",
    "id":

In [6]:
# Sub-key frequency across 100 docs for each dict-type nested field
DICT_NESTED_KEYS = ["miniProfile", "currentPosition", "memberDistance", "email"]

for key in DICT_NESTED_KEYS:
    sub_counts = Counter()
    key_present = 0
    for doc in freq_docs:
        val = doc.get(key)
        if isinstance(val, dict):
            sub_counts.update(val.keys())
            key_present += 1
    if key_present == 0:
        print(f"\n[{key}] — not a dict in any sampled doc")
        continue
    rows = [(sk, cnt, f"{100*cnt/key_present:.0f}%") for sk, cnt in sub_counts.most_common()]
    sub_df = pd.DataFrame(rows, columns=["sub-key", "count", "presence"])
    print(f"\n[{key}]  (present in {key_present}/{total} docs)")
    print(sub_df.to_string(index=False))


[miniProfile]  (present in 100/100 docs)
                sub-key  count presence
               lastName    100     100%
 customLastNameOutdated    100     100%
              createdAt    100     100%
                 avatar    100     100%
        customFirstName    100     100%
                     id    100     100%
customFirstNameOutdated    100     100%
               personId    100     100%
       originalLastName    100     100%
         customLastName    100     100%
               headline    100     100%
      originalFirstName    100     100%
              firstName    100     100%
    originalSentAtToPAS     97      97%
              updatedAt      4       4%

[currentPosition]  (present in 97/100 docs)
               sub-key  count presence
customPositionOutdated     97     100%
               company     97     100%
      originalPosition     97     100%
                    id     97     100%
              personId     97     100%
         customCompany     97     100%


## 4. Sample `analysis` collection — keys and value distributions

In [7]:
analysis_docs = [doc.to_dict() for doc in db.collection("analysis").limit(FREQ_SAMPLE).stream()]
total_a = len(analysis_docs)
print(f"Fetched {total_a} docs from 'analysis'")

# Key frequency
a_key_counts = Counter()
for doc in analysis_docs:
    a_key_counts.update(doc.keys())

a_freq_df = pd.DataFrame(
    [(k, v, f"{100*v/total_a:.1f}%") for k, v in a_key_counts.most_common()],
    columns=["key", "count", "presence"],
)
a_freq_df

Fetched 100 docs from 'analysis'


,key,count,presence
0,memberDistance,100,100.0%
1,lh_id,100,100.0%
2,summary,100,100.0%
3,profileUrl,100,100.0%
4,industry,93,93.0%
5,function,93,93.0%
6,seniority,93,93.0%


In [8]:
# Value distributions for categorical fields
analysis_df = pd.DataFrame(analysis_docs)

for col in ["industry", "function", "seniority"]:
    if col not in analysis_df.columns:
        print(f"\n[{col}] — column not present")
        continue
    counts = analysis_df[col].value_counts(dropna=False)
    print(f"\n--- {col} ({counts.sum()} docs, {analysis_df[col].isna().sum()} missing) ---")
    print(counts.to_string())


--- industry (100 docs, 7 missing) ---
industry
Healthcare                59
NaN                        7
Biotechnology              7
Technology                 5
Life Sciences              3
Education                  3
Pharmaceuticals            2
Unknown                    2
Staffing & Recruiting      1
Public Policy              1
Nonprofit                  1
Manufacturing              1
Supply Chain               1
Fitness                    1
Information Technology     1
Insurance                  1
Medical Devices            1
Finance                    1
Business Services          1
Consulting                 1

--- function (100 docs, 7 missing) ---
function
Operations                                       17
Sales                                            15
NaN                                               7
Marketing                                         4
Executive Management                              3
Business Development                              3
General Ma

In [9]:
# Show a preview of the analysis DataFrame (first 10 rows, drop long text columns)
preview_cols = [c for c in analysis_df.columns if c != "summary"]
analysis_df[preview_cols].head(10)

,memberDistance,lh_id,profileUrl,industry,function,seniority
0,3,3294,https://www.linkedin.com/in/-jananova-/,NaN,NaN,NaN
1,2,23757,https://www.linkedin.com/in/007rcm/,Healthcare,Operations,Senior
2,2,21616,https://www.linkedin.com/in/060360ymlouis/,NaN,NaN,NaN
3,2,16649,https://www.linkedin.com/in/0karthik0/,NaN,NaN,NaN
4,2,23553,https://www.linkedin.com/sales/people/ACwAADui...,Healthcare,Sales,Executive
5,2,24421,https://www.linkedin.com/sales/people/ACwAAAX3...,Healthcare,Executive Management,Executive
6,2,23934,https://www.linkedin.com/sales/people/ACwAAACY...,Staffing & Recruiting,Recruiting & Business Development,Mid
7,2,23821,https://www.linkedin.com/sales/people/ACwAADvD...,Healthcare,Sales,Senior
8,2,24439,https://www.linkedin.com/sales/people/ACwAAAX6...,Life Sciences,Business Development,Executive
9,2,23480,https://www.linkedin.com/sales/people/ACwAAAX7...,Biotechnology,Founding/Executive,C-Level
